In [11]:
# transformer-lens 3.4.0 pins torchvision>=0.22,<0.23 (for torch 2.7.x),
# but Colab has torch 2.10 + torchvision 0.25. This breaks pip's resolver.
# Workaround: install both with --no-deps, then install dependencies separately.

# Step 1: Install sae-lens and transformer-lens without the torchvision constraint
!pip install --no-deps transformer-lens sae-lens

# Step 2: Install all actual dependencies (minus torchvision, not needed for text SAEs)
!pip install \
    "transformers>=5.4.0,<6.0.0" \
    "datasets>=3.1.0" \
    "einops>=0.6.0" \
    "jaxtyping>=0.2.11" \
    "fancy-einsum>=0.0.3" \
    "beartype>=0.14.1" \
    "accelerate>=0.23.0" \
    "rich>=12.6.0" \
    "sentencepiece" \
    "typeguard>=4.2,<5" \
    "wandb>=0.13.5" \
    "transformers-stream-generator>=0.0.5,<0.1" \
    "better-abc>=0.0.3" \
    "protobuf>=3.20.0" \
    "huggingface-hub>=0.23.2" \
    "safetensors>=0.4.2,<1.0.0" \
    "plotly>=5.19.0" \
    "nltk>=3.8.1" \
    "python-dotenv>=1.0.1" \
    "pyyaml>=6.0.1" \
    "simple-parsing>=0.1.8" \
    "tenacity>=9.0.0" \
    "babe>=0.0.7"

In [ ]:
from huggingface_hub import login
import os
from __future__ import annotations

import pandas as pd
import argparse
import re
import sys
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import torch, gc
from sae_lens import SAE
from transformer_lens import HookedTransformer

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive"
GIT_DIR = "/content/SAE_Surrogate"

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.mem_get_info())
print(torch.cuda.is_bf16_supported())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
0
NVIDIA A100-SXM4-40GB
True


In [ ]:
with open(f"{DRIVE_DIR}/tokens/hftoken.txt") as f:
    token = f.readline().strip()

login(token=token)

In [ ]:
!git clone https://github.com/ntjohn04/SAE_Surrogate

sys.path.append(f"{GIT_DIR}")

import util_neuronpedia

Cloning into 'SAE_Surrogate'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 22 (delta 6), reused 19 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 400.34 KiB | 20.02 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [48]:
DEFAULT_RELEASE = "gemma-scope-2b-pt-res"
DEFAULT_SAE_ID = "layer_20/width_16k/average_l0_71"
DEFAULT_MODEL = "gemma-2-2b"
NEURONPEDIA_FEATURE_API = "https://www.neuronpedia.org/api/feature"

DEVICE = 'cuda'
DTYPE = torch.bfloat16


In [ ]:
print("----loading SAE----")
sae = SAE.from_pretrained(
    release=DEFAULT_RELEASE,
    sae_id=DEFAULT_SAE_ID,
    device=DEVICE,
    dtype=DTYPE
)

print("----loading feature catalog----")
if not os.path.exists(f"{GIT_DIR}/feature_catalog.csv"):
    print("\tbuilding feature catalog...")
    feature_catalog = util_neuronpedia.build_feature_catalog()
    feature_catalog.to_csv(f"{GIT_DIR}/feature_catalog.csv", index=False)
else:
    feature_catalog = pd.read_csv(f"{GIT_DIR}/feature_catalog.csv")

print("----loading hooks----")
metadata = getattr(sae.cfg, "metadata", None)
hook_name = getattr(metadata, "hook_name", None)
match = re.search(r"blocks\.(\d+)\.", hook_name)
hook_layer = int(match.group(1)) if match else None

model_name = getattr(metadata, "model_name", None)
prepend_bos = getattr(metadata, "prepend_bos", None)

print(f"\tHook name:\t{hook_name}")
print(f"\tHook layer:\t{hook_layer}")
print(f"\tModel name:\t{model_name}")
print(f"\tPrepend BOS:\t{prepend_bos}")

print("----loading model----")
model = HookedTransformer.from_pretrained_no_processing(model_name,
                                          device=DEVICE, 
                                          dtype=DTYPE, 
                                          default_prepend_bos=prepend_bos, 
                                          first_n_layers=hook_layer+1)

----loading SAE----
----loading hooks----
	Hook name:	blocks.20.hook_resid_post
	Hook layer:	20
	Model name:	gemma-2-2b
	Prepend BOS:	True
----loading model----


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded pretrained model gemma-2-2b into HookedTransformer


In [35]:
def get_feature_sequence(sae, model, prepend_bos, hook_name, top_k, prompt):
    with torch.inference_mode():
        tokens = model.to_tokens(prompt, prepend_bos=prepend_bos)
        str_tokens = model.to_str_tokens(tokens[0])

        _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
        activations = cache[hook_name]
        feature_acts = sae.encode(activations)

    return feature_acts, str_tokens

In [49]:
feature_acts, str_tokens = get_feature_sequence(sae, model, prepend_bos, hook_name, 5, "USA France London Australia Russia")